# HDB Resale Pipeline — Big Data Concepts

One data lifecycle, end to end, on **real Singapore HDB resale transactions**
from [data.gov.sg](https://data.gov.sg).

```
EXTRACT → STORE → SERVE → CACHE
```

The order matters: get it working, watch it be slow, *then* reach for Redis.
A cache introduced before there is a slow query is just a word.

| Step | What it covers |
|---|---|
| 1. Extract | Live API, paging, rate limits, landing a file |
| 2. Store | MongoDB as the system of record |
| 3. Serve | Become an API instead of just consuming one |
| 4. Cache | Redis as the speed layer — same answer, less cost |


In [1]:
import sys
sys.path.insert(0, '../src')   # run this notebook from notebooks/

---
## Step 1 — Extract

The dataset is somebody else's, it is live, and it will not hand us everything at once.

In [2]:
from step1_extract import fetch_resale

# A small live pull, just to watch the paging happen.
sample = fetch_resale(100, page_size=50)
sample.head()

  238,573 rows available; taking 2 pages of 50, one every 119,286 rows


  page  1/2 @offset=0       +50    total 50


  page  2/2 @offset=119286  +50    total 100


,flat_id,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,price_per_sqm
0,1,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,232000.0,5272.73
1,2,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,250000.0,3731.34
2,3,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,262000.0,3910.45
3,4,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,265000.0,3897.06
4,5,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,265000.0,3955.22


Three things a local CSV would never have taught us:

1. **You page.** The API caps each response, so you loop with `limit`/`offset`.
2. **It pushes back.** Page too fast and you get `429 Too Many Requests`, so
   the client backs off and retries.
3. **Order is not random.** The rows arrive sorted by town — take the first
   100 and you get one town. We spread our pages across the whole dataset
   instead, which is why `fetch_resale` is more than one line.

In [3]:
# The full dataset was landed once and lives on disk - no network needed.
from step1_extract import read_local

df = read_local()
print(f"{len(df):,} rows, {df['town'].nunique()} towns, {df['flat_type'].nunique()} flat types")
df.head()

24,000 rows, 26 towns, 7 flat types


,flat_id,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,price_per_sqm
0,1,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,232000.0,5272.73
1,2,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,250000.0,3731.34
2,3,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,262000.0,3910.45
3,4,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,265000.0,3897.06
4,5,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,265000.0,3955.22


Extraction ends the moment the data is **landed as a file**. Everything after
this point reads that file, not the API. That boundary is the whole reason
pipelines are restartable.

---
## Step 2 — Store it in MongoDB

The system of record: durable, queryable, the thing everything downstream trusts.

In [4]:
from step2_load_mongo import load_flats, list_towns, read_flats

count = load_flats(df)
print(f"MongoDB now holds {count:,} transactions across {len(list_towns())} towns")
read_flats(limit=1)[0]

MongoDB now holds 24,000 transactions across 26 towns


{'flat_id': 1,
 'month': '2017-01',
 'town': 'ANG MO KIO',
 'flat_type': '2 ROOM',
 'block': '406',
 'street_name': 'ANG MO KIO AVE 10',
 'storey_range': '10 TO 12',
 'floor_area_sqm': 44.0,
 'flat_model': 'Improved',
 'lease_commence_date': 1979,
 'resale_price': 232000.0,
 'price_per_sqm': 5272.73}

In [5]:
# Mongo answers questions the CSV cannot, without loading everything first.
from step2_load_mongo import town_summary

town_summary('BEDOK')

{'town': 'BEDOK',
 'transactions': 1129,
 'avg_price': 520385.59,
 'min_price': 188000.0,
 'max_price': 1270000.0,
 'avg_price_per_sqm': 5287.39,
 'elapsed_ms': 11.25,
 'source': 'mongodb'}

Note `elapsed_ms` in that result. Keep an eye on it — it is the number Step 4
is going to attack.

---
## Step 3 — Serve it back out

We consumed an API in Step 1. Now we become one.

The API is a server, so it needs **its own terminal**:

```bash
./run.sh api
```

Leave it running, then carry on here.

In [6]:
import requests
BASE = 'http://localhost:5001'

requests.get(f'{BASE}/', timeout=10).json()

{'endpoints': {'GET /flats': 'resale transactions; ?town=&flat_type=&limit=',
  'GET /flats/<flat_id>': 'a single transaction',
  'GET /health': 'mongo + redis liveness',
  'GET /overview': 'every town ranked; add ?cache=true',
  'GET /stats': 'dataset-level summary',
  'GET /towns': 'every town we hold',
  'GET /towns/<town>': 'aggregated stats; add ?cache=true for Step 4'},
 'service': 'hdb-resale-pipeline'}

In [7]:
# This is Step 1's code shape with only the URL changed.
# That is the whole point: extraction is a role, not a step.
flats = requests.get(f'{BASE}/flats', params={'town': 'BEDOK', 'limit': 5}, timeout=30).json()
for f in flats:
    print(f"{f['flat_type']:<11} {f['floor_area_sqm']:>5.0f} sqm  ${f['resale_price']:>10,.0f}")

2 ROOM         45 sqm  $   238,000
3 ROOM         68 sqm  $   272,000
3 ROOM         59 sqm  $   278,000
3 ROOM         68 sqm  $   280,000
3 ROOM         59 sqm  $   280,000


### And a dashboard is just another consumer

```bash
./run.sh dashboard
```

[src/dashboard.py](../src/dashboard.py) calls the exact same `requests.get`.
It touches neither Mongo nor Redis — the API already does that. Three
consumers now: `curl`, this notebook, and Streamlit.

---
## Step 4 — Cache it with Redis

Now the slow part. Ask for the **market overview** — every town ranked by
price per sqm. There is no `$match` to narrow it down, so Mongo walks the
entire collection and sorts the result. It is also the query a dashboard
homepage runs for every single visitor.

In [8]:
from step2_load_mongo import market_overview

overview = market_overview()
print(f"{overview['elapsed_ms']} ms from {overview['source']}")
overview['towns'][:3]

27.45 ms from mongodb


[{'transactions': 301,
  'avg_price': 680915.75,
  'avg_price_per_sqm': 8304.82,
  'town': 'CENTRAL AREA'},
 {'transactions': 1131,
  'avg_price': 727477.1,
  'avg_price_per_sqm': 7803.0,
  'town': 'KALLANG/WHAMPOA'},
 {'transactions': 742,
  'avg_price': 634044.21,
  'avg_price_per_sqm': 7610.37,
  'town': 'QUEENSTOWN'}]

In [9]:
from step4_cache_redis import demo_overview

demo_overview()

Market overview - every town, ranked by price per sqm
  miss:    24.81 ms  (mongodb walks 26 towns)
  hit:      1.26 ms  (redis)     ~20x faster
  priciest: CENTRAL AREA at $8,305/sqm


({'towns': [{'transactions': 301,
    'avg_price': 680915.75,
    'avg_price_per_sqm': 8304.82,
    'town': 'CENTRAL AREA'},
   {'transactions': 1131,
    'avg_price': 727477.1,
    'avg_price_per_sqm': 7803.0,
    'town': 'KALLANG/WHAMPOA'},
   {'transactions': 742,
    'avg_price': 634044.21,
    'avg_price_per_sqm': 7610.37,
    'town': 'QUEENSTOWN'},
   {'transactions': 818,
    'avg_price': 782974.85,
    'avg_price_per_sqm': 7311.9,
    'town': 'BISHAN'},
   {'transactions': 84,
    'avg_price': 777007.52,
    'avg_price_per_sqm': 7133.58,
    'town': 'BUKIT TIMAH'},
   {'transactions': 395,
    'avg_price': 605464.12,
    'avg_price_per_sqm': 6813.86,
    'town': 'MARINE PARADE'},
   {'transactions': 497,
    'avg_price': 558285.95,
    'avg_price_per_sqm': 6471.54,
    'town': 'BUKIT MERAH'},
   {'transactions': 867,
    'avg_price': 530329.45,
    'avg_price_per_sqm': 6004.15,
    'town': 'TOA PAYOH'},
   {'transactions': 1247,
    'avg_price': 589726.55,
    'avg_price_per_sq

Same answer. A fraction of the cost. That is the entire idea of a cache:

```
look in Redis  →  hit?  return it
               →  miss? ask MongoDB, remember the answer, return it
```

### When *not* to cache

Run the same comparison on a single town, where Mongo has an index and only
touches ~1,000 documents:

In [10]:
from step4_cache_redis import demo

demo('BEDOK')

  CACHE MISS town:BEDOK -> aggregating in MongoDB
  CACHE HIT  town:BEDOK
  miss:     4.42 ms  (mongodb)
  hit:      1.07 ms  (redis)     ~4x faster
  same answer either way: avg $520,386 over 1,129 transactions


({'town': 'BEDOK',
  'transactions': 1129,
  'avg_price': 520385.59,
  'min_price': 188000.0,
  'max_price': 1270000.0,
  'avg_price_per_sqm': 5287.39,
  'elapsed_ms': 4.42,
  'source': 'mongodb'},
 {'town': 'BEDOK',
  'transactions': 1129,
  'avg_price': 520385.59,
  'min_price': 188000.0,
  'max_price': 1270000.0,
  'avg_price_per_sqm': 5287.39,
  'elapsed_ms': 1.07,
  'source': 'redis'})

Barely a win. **A cache is only worth it when the thing behind it is
expensive.** Put one in front of a fast indexed lookup and you have added a
second system to keep consistent in exchange for almost nothing.

Worth thinking about: the TTL is 60 seconds. If a new transaction lands at
second 5, nobody sees it for another 55. Who is allowed to see a stale
answer, and for how long? That question has no universal answer — it is a
decision you make per query.

---
## The thread

```
EXTRACT → STORE → SERVE → CACHE
```

- **Extraction is a role, not a step.** We pulled from data.gov.sg, then
  served our own API, and something else pulled from us. Same verb, different
  side of the request.
- **Each layer exists to fix a specific pain.** Mongo because a CSV cannot
  answer questions. Redis because the aggregation was slow. Nothing here was
  added because it was fashionable.
- **The answer never changed.** Only the cost of getting it did. That is most
  of what data engineering is.

### Try it yourself

Run `./run.sh api` in one terminal and `./run.sh consume` in another.

`consume_own_api.py` is Step 1's code with nothing changed but the URL. In
Step 1 you pulled from data.gov.sg; now you are pulling from something you
built, and the code cannot tell the difference.